# Benchmark de Transformación: Tradicional (PyMuPDF) vs IA (Docling)

Este notebook compara dos enfoques fundamentalmente distintos para la extracción de datos de PDF:
1. **PyMuPDF (fitz)**: Un motor tradicional, extremadamente rápido, que extrae texto plano basándose en coordenadas y capas del documento.
2. **Docling (IBM)**: Un motor de nueva generación que utiliza modelos de IA para comprender la estructura, tablas y jerarquía, exportando a Markdown.

El objetivo es evaluar el balance entre **velocidad** (PyMuPDF) y **calidad estructural** (Docling) para el procesamiento de informes del Congreso.

In [ ]:
import os, sys, time, re, fitz, pymupdf4llm
from pathlib import Path
from IPython.display import display, Markdown
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from chonkie import SemanticChunker


## 1. Configuración
Define aquí la ruta de tu informe del Congreso.

In [ ]:
# === CONFIGURACIÓN ===
PDF_PATH = os.path.join("inputs", "test_2026.pdf")  # paper de arXiv para testear el benchmark
OUTPUT_DIR = "outputs"
CHUNKING_DIR = "outputs-chunking"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHUNKING_DIR, exist_ok=True)

# Diagnóstico de ruta (Windows + Python + Docling/Torch)
if any(ord(c) > 127 for c in os.path.abspath(".")):
    print("⚠️ ADVERTENCIA: La ruta actual contiene caracteres no-ASCII (acentos, ñ, etc).")
    print("Esto puede causar fallos en Docling/PyTorch en Windows.")
    print("Se recomienda renombrar la carpeta 'Construcción' a 'Construccion'.\n")

## 2. Funciones de Extracción

In [ ]:
def process_with_pymupdf(pdf_path, output_folder):
    print("🚀 Procesando con PyMuPDF (Método Tradicional)...\n")
    start_time = time.time()
    text_content = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text_content.append(page.get_text("text"))
    full_text = "\n".join(text_content)
    duration = time.time() - start_time
    pdf_name = Path(pdf_path).stem
    output_path = os.path.join(output_folder, f"{pdf_name}_salida_pymupdf.txt")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"✅ PyMuPDF finalizado en {duration:.4f}s")
    return full_text, duration

def process_with_docling(pdf_path, output_folder):
    print("🚀 Procesando con Docling (Método IA)...\n")
    converter = DocumentConverter()
    start_time = time.time()
    result = converter.convert(pdf_path)
    markdown_content = result.document.export_to_markdown()
    duration = time.time() - start_time
    pdf_name = Path(pdf_path).stem
    output_path = os.path.join(output_folder, f"{pdf_name}_salida_docling.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)
    print(f"✅ Docling finalizado en {duration:.2f}s")
    return result, markdown_content, duration

def process_with_pymupdf4llm(pdf_path, output_folder):
    print("🚀 Procesando con PyMuPDF4LLM (Optimizado para LLM)...\n")
    start_time = time.time()
    md_text = pymupdf4llm.to_markdown(pdf_path)
    duration = time.time() - start_time
    pdf_name = Path(pdf_path).stem
    output_path = os.path.join(output_folder, f"{pdf_name}_salida_pymupdf4llm.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(md_text)
    print(f"✅ PyMuPDF4LLM finalizado en {duration:.4f}s")
    return md_text, duration

def process_with_chonkie(text):
    if not text or len(text.strip()) == 0:
        return [], 0.0
    chunker = SemanticChunker(embedding_model="all-MiniLM-L6-v2", threshold=0.5, chunk_size=512)
    start_time = time.time()
    chunks = chunker.chunk(text)
    duration = time.time() - start_time
    return chunks, duration


## 2.1 Métricas de Calidad

Computamos dos grupos de métricas sobre el texto extraído:

- **Calidad del texto** (`chars`, `words`, `broken_hyphens`): mide propiedades del texto en sí — es justa para ambos motores. `broken_hyphens` cuenta artefactos tipo `palabra-\npalabra` que un buen extractor debería resolver (menos = mejor).
- **Estructura** (`headings`, `table_rows`, `list_items`): cuenta elementos sintácticos de Markdown. Está sesgada por diseño: sobre la salida plana de PyMuPDF tiende a 0; sobre el Markdown de Docling refleja la estructura recuperada.

In [ ]:
# Métricas estructurales (sesgan a Markdown) + métricas de texto (fair a ambos motores).
HEADING_RE = re.compile(r"^#{1,6}\s")
TABLE_ROW_RE = re.compile(r"^\|.*\|\s*$")
LIST_RE = re.compile(r"^[-*+]\s")
WORD_RE = re.compile(r"\w+", re.UNICODE)
BROKEN_HYPHEN_RE = re.compile(r"\w-\n\w")  # palabra-\npalabra: artefacto de PDF que un buen extractor une

def compute_quality_metrics(text):
    headings = table_rows = list_items = 0
    for raw_line in text.splitlines():
        line = raw_line.lstrip()
        if HEADING_RE.match(line):
            headings += 1
        if TABLE_ROW_RE.match(line):
            table_rows += 1
        if LIST_RE.match(line):
            list_items += 1
    return {
        # Calidad del texto (fair a ambos motores)
        "chars": len(text),
        "words": len(WORD_RE.findall(text)),
        "broken_hyphens": len(BROKEN_HYPHEN_RE.findall(text)),  # menos = mejor
        # Estructura (sólo Docling produce Markdown)
        "headings": headings,
        "table_rows": table_rows,
        "list_items": list_items,
    }

## 3. Ejecución del Benchmark
Compara el rendimiento y los resultados.

In [ ]:
text_py, time_py = "", 0
text_doc, time_doc = "", 0
text_4llm, time_4llm = "", 0
result_doc = None
chunks_py, c_time_py = [], 0.0
chunks_doc, c_time_doc = [], 0.0
chunks_doc_native, c_time_doc_native = [], 0.0
chunks_4llm, c_time_4llm = [], 0.0

if os.path.exists(PDF_PATH):
    # 1. PyMuPDF
    try:
        text_py, time_py = process_with_pymupdf(PDF_PATH, OUTPUT_DIR)
        if len(text_py.strip()) == 0:
            print("⚠️ PyMuPDF no extrajo texto. El PDF podría ser una imagen/escaneo.")
        else:
            chunks_py, c_time_py = process_with_chonkie(text_py)
    except Exception as e:
        print(f"❌ Error en PyMuPDF: {e}")

    # 2. PyMuPDF4LLM
    try:
        text_4llm, time_4llm = process_with_pymupdf4llm(PDF_PATH, OUTPUT_DIR)
        if len(text_4llm.strip()) > 0:
            chunks_4llm, c_time_4llm = process_with_chonkie(text_4llm)
    except Exception as e:
        print(f"❌ Error en PyMuPDF4LLM: {e}")

    print("-" * 30)

    # 3. Docling + Chunking Nativo + Chonkie
    try:
        result_doc, text_doc, time_doc = process_with_docling(PDF_PATH, OUTPUT_DIR)
        # Chunking semántico con Chonkie
        chunks_doc, c_time_doc = process_with_chonkie(text_doc)
        # Chunking nativo de Docling (HybridChunker)
        start_c_native = time.time()
        doc_chunker = HybridChunker()
        chunks_doc_native = list(doc_chunker.chunk(result_doc.document))
        c_time_doc_native = time.time() - start_c_native
    except Exception as e:
        print(f"❌ Error en Docling: {e}")
        print("Tip: Verifica que no haya acentos en la ruta de las carpetas.")

    # === RESUMEN DE RENDIMIENTO ===
    print(f"\n{'='*60}")
    print(f"  RESUMEN DE RENDIMIENTO")
    print(f"{'='*60}")
    print(f"{'Método':<20}{'Extracción':>12}{'Chunking':>12}{'Total':>12}")
    print(f"{'PyMuPDF':<20}{time_py:>11.4f}s{c_time_py:>11.4f}s{time_py+c_time_py:>11.4f}s")
    print(f"{'PyMuPDF4LLM':<20}{time_4llm:>11.4f}s{c_time_4llm:>11.4f}s{time_4llm+c_time_4llm:>11.4f}s")
    print(f"{'Docling':<20}{time_doc:>11.2f}s{c_time_doc:>11.4f}s{time_doc+c_time_doc:>11.2f}s")
    if time_py > 0 and time_doc > 0:
        print(f"\nFactor velocidad (Extracción): {time_doc/time_py:.1f}x más rápido PyMuPDF")

    # === CALIDAD Y CHUNKING ===
    m_py = compute_quality_metrics(text_py) if text_py else {"chars":0,"words":0,"broken_hyphens":0,"headings":0,"table_rows":0,"list_items":0}
    m_doc = compute_quality_metrics(text_doc) if text_doc else {"chars":0,"words":0,"broken_hyphens":0,"headings":0,"table_rows":0,"list_items":0}
    m_4llm = compute_quality_metrics(text_4llm) if text_4llm else {"chars":0,"words":0,"broken_hyphens":0,"headings":0,"table_rows":0,"list_items":0}

    print(f"\n{'='*60}")
    print(f"  CALIDAD DEL TEXTO")
    print(f"{'='*60}")
    print(f"{'Métrica':<20}{'PyMuPDF':>12}{'PyMuPDF4LLM':>15}{'Docling':>12}")
    for key in ("chars", "words", "broken_hyphens"):
        print(f"{key:<20}{m_py[key]:>12}{m_4llm[key]:>15}{m_doc[key]:>12}")

    print(f"\n{'='*60}")
    print(f"  CHUNKING COMPARATIVO")
    print(f"{'='*60}")
    print(f"{'Métrica':<25}{'PyMuPDF':>12}{'PyMuPDF4LLM':>15}{'Docling':>12}")
    print(f"{'Chunks (Chonkie)':<25}{len(chunks_py):>12}{len(chunks_4llm):>15}{len(chunks_doc):>12}")
    print(f"{'Chunks (Docling Nativo)':<25}{'-':>12}{'-':>15}{len(chunks_doc_native):>12}")
    print(f"{'Latencia Chonkie':<25}{c_time_py:>11.4f}s{c_time_4llm:>14.4f}s{c_time_doc:>11.4f}s")
    print(f"{'Latencia Docling Nativo':<25}{'-':>12}{'-':>15}{c_time_doc_native:>11.4f}s")

    print(f"\n{'='*60}")
    print(f"  ESTRUCTURA (sesgada a Markdown)")
    print(f"{'='*60}")
    print(f"{'Métrica':<20}{'PyMuPDF':>12}{'PyMuPDF4LLM':>15}{'Docling':>12}")
    for key in ("headings", "table_rows", "list_items"):
        print(f"{key:<20}{m_py[key]:>12}{m_4llm[key]:>15}{m_doc[key]:>12}")
else:
    print(f"❌ No se encontró: {PDF_PATH}")


## 4. Contraste de Resultados
Visualiza cómo cada herramienta interpreta el mismo contenido.

In [ ]:
print("### [TRADICIONAL] SALIDA PYMUPDF (Fragmento) ###")
if text_py:
    print(text_py[:1500] + "...")
else:
    print("(No se pudo extraer texto con PyMuPDF)")

print("\n" + "="*60 + "\n")

print("### [IA] SALIDA DOCLING (Fragmento Markdown) ###")
if text_doc:
    display(Markdown(text_doc[:1500] + "..."))
else:
    print("(No se pudo extraer texto con Docling)")

print("\n" + "="*60 + "\n")

print("### [LLM] SALIDA PYMUPDF4LLM (Fragmento Markdown) ###")
if text_4llm:
    display(Markdown(text_4llm[:1500] + "..."))
else:
    print("(No se pudo extraer texto con PyMuPDF4LLM)")

## 5. Inspección de Chunks (Docling + Chonkie)
Usa esta celda para ver exactamente dónde decidió cortar Chonkie basándose en el cambio de significado (Semantic Shift).

In [ ]:
def export_chunks_to_md(chunks, base_filename, output_folder):
    output_path = os.path.join(output_folder, f"{base_filename}_chunks.md")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"# Chunks Generados por Chonkie - {base_filename}\n\n")
        for i, chunk in enumerate(chunks):
            content = chunk.text if hasattr(chunk, 'text') else str(chunk)
            f.write(f"## --- FRAGMENTO {i+1} ---\n")
            f.write(content.strip() + "\n\n")
            f.write("---\n\n")
    return output_path

print("💾 Exportando resultados a archivos Markdown...\n")

if 'chunks_doc' in locals() and chunks_doc:
    path_chunks = export_chunks_to_md(chunks_doc, "docling_chonkie", CHUNKING_DIR)
    
    # Exportar Chunking Nativo de Docling
    if 'chunks_doc_native' in locals() and chunks_doc_native:
        path_native = export_chunks_to_md(chunks_doc_native, "docling_nativo", CHUNKING_DIR)
        print(f"✅ Chunks Nativos de Docling guardados en: {path_native}")
    print(f"✅ Chunks de Chonkie (sobre Docling) guardados en: {path_chunks}")
    
    # El archivo de docling original ya se guarda en la función process_with_docling
    # pero vamos a recordar su ubicación para el usuario
    pdf_name = Path(PDF_PATH).stem
    print(f"✅ Texto original de Docling disponible en: {OUTPUT_DIR}/{pdf_name}_salida_docling.md")
else:
    print("⚠️ No hay resultados de Docling para exportar.")
